### Logit Model
- this is built using the model_data
- 80/20 split using 6001 seed

In [61]:
import pandas as pd
import numpy as np 
import plotly.express as px
import plotly.graph_objects as go 
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.metrics import log_loss, confusion_matrix, accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve
import plotly.graph_objects as go
from sklearn.model_selection import StratifiedKFold, GridSearchCV


px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Dark24

In [9]:
# load in the model data 
train = pd.read_csv("model_data/restaurants_model_development.csv")
test = pd.read_csv("model_data/restaurants_model_holdout.csv")

In [10]:
display(train.shape)
train.head()

(41802, 54)

,business_id,split,name,city,postal_code,review_count,is_open,stars,state,latitude,...,open_weekend,late_night,early_open,cuisine,n_cuisine_groups,service_format,category_count,chain_size,is_chain,price_range
0,MTSW4McQd7CbVtyjqoe9mw,development,St Honore Pastries,Philadelphia,19107,80,1,4.0,PA,39.955505,...,Yes,No,Yes,Breakfast & Cafe,2,Standard,5,1,No,1
1,CF33F8-E6oudUQ46HnavjQ,development,Sonic Drive-In,Ashland City,37015,6,1,2.0,TN,36.269593,...,Yes,Yes,Yes,Desserts & Snacks,3,Fast food,6,113,Yes,1
2,il_Ro8jwPlHresjw9EGmBg,development,Denny's,Indianapolis,46227,28,1,2.5,IN,39.637133,...,Yes,Yes,Yes,American,2,Standard,4,81,Yes,1
3,MUTTqe8uqyMdBl186RmNeA,development,Tuna Bar,Philadelphia,19106,245,1,4.0,PA,39.953949,...,Yes,Yes,No,Japanese,1,Standard,3,1,No,2
4,ROeacJQwBeh05Rqg7F6TCg,development,BAP,Philadelphia,19147,205,1,4.5,PA,39.943223,...,Yes,No,No,Korean,1,Standard,2,1,No,1


In [11]:
train.columns

Index(['business_id', 'split', 'name', 'city', 'postal_code', 'review_count',
       'is_open', 'stars', 'state', 'latitude', 'longitude', 'takeout',
       'delivery', 'outdoor_seating', 'reservations', 'has_tv',
       'good_for_groups', 'good_for_kids', 'accepts_credit_cards',
       'bike_parking', 'caters', 'table_service', 'wheelchair_accessible',
       'happy_hour', 'dogs_allowed', 'drive_thru', 'alcohol', 'wifi',
       'noise_level', 'attire', 'price_level', 'parking_any', 'parking_lot',
       'parking_street', 'ambience', 'good_for_breakfast', 'good_for_brunch',
       'good_for_lunch', 'good_for_dinner', 'good_for_latenight',
       'good_for_dessert', 'listed_days', 'weekly_hours', 'avg_daily_hours',
       'open_weekend', 'late_night', 'early_open', 'cuisine',
       'n_cuisine_groups', 'service_format', 'category_count', 'chain_size',
       'is_chain', 'price_range'],
      dtype='str')

In [ ]:
price_vs = train[['price_range','price_level']]
price_vs.head()
price_vs.dtypes

price_range        str
price_level    float64
dtype: object

- The steps aren’t even
- Going from 1 to 2 adds 0.29 stars, 2 to 3 adds only 0.09, and 3 to 4 actually drops slightly (though with just 149 restaurants at level 4, that dip is mostly noise). 
- A single numeric coefficient would force one fixed effect per step, which would overstate the higher levels and understate the jump from 1 to 2. 
- decision: price_range (categorical)

In [17]:
train.groupby('price_range')['stars'].agg(['size', 'mean']).round(2)

,size,mean
price_range,,
1,15593,3.30
2,18494,3.59
3,1331,3.68
4,149,3.64
Unknown,6235,3.80


In [ ]:
y = train["stars"]
X = train[['state', 'latitude', 'longitude', 'takeout',
       'delivery', 'outdoor_seating', 'reservations', 'has_tv',
       'good_for_groups', 'good_for_kids', 'accepts_credit_cards',
       'bike_parking', 'caters', 'table_service', 'wheelchair_accessible',
       'happy_hour', 'dogs_allowed', 'drive_thru', 'alcohol', 'wifi',
       'noise_level', 'attire', 'parking_any', 'parking_lot',
       'parking_street','ambience', 'good_for_breakfast', 'good_for_brunch',
       'good_for_lunch', 'good_for_dinner', 'good_for_latenight',
       'good_for_dessert', 'listed_days', 'weekly_hours', 'avg_daily_hours',
       'open_weekend', 'late_night', 'early_open', 'cuisine',
       'n_cuisine_groups', 'service_format', 'category_count', 'chain_size',
       'is_chain', 'price_range']]

cats = ["state", "cuisine", "service_format", "is_chain", "price_range",
        "takeout", "delivery", "outdoor_seating", "reservations", "has_tv",
        "good_for_groups", "good_for_kids", "accepts_credit_cards", "bike_parking",
        "caters", "table_service", "wheelchair_accessible", "happy_hour",
        "dogs_allowed", "drive_thru", "alcohol", "wifi", "noise_level", "attire",
        "parking_any", "parking_lot", "parking_street", "ambience",
        "good_for_breakfast", "good_for_brunch", "good_for_lunch", "good_for_dinner",
        "good_for_latenight", "good_for_dessert",
        "open_weekend", "late_night", "early_open"]

nums = ["latitude", "longitude", "listed_days", "weekly_hours", "avg_daily_hours",
        "n_cuisine_groups", "category_count", "chain_size"]

X_train = train[cats + nums]
y_train = (train["stars"] >= 4).astype(int)

X_test = test[cats + nums]
y_test = (test["stars"] >= 4).astype(int)


In [28]:
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="infrequent_if_exist",
                              min_frequency=50), cats),
    ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                              StandardScaler()), nums),
])

In [29]:
logreg = LogisticRegression(max_iter=5000, random_state=123)
pipe = Pipeline(steps=[("preprocess", preprocess), ("model", logreg)])
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](45,)","['state','cuisine','service_format',...,'n_cuisine_groups', 'category_count','chain_size']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,45
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remain

In [30]:
p_train = pipe.predict_proba(X_train)[:, 1]
p_test = pipe.predict_proba(X_test)[:, 1]

log_odds_train = pipe.decision_function(X_train)
log_odds_test = pipe.decision_function(X_test)
threshold = 0.5

In [32]:
model = pipe.named_steps["model"]
feature_names = pipe.named_steps["preprocess"].get_feature_names_out()

print("Intercept:", model.intercept_[0])

coef = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model.coef_[0],
    "Odds Ratio": np.exp(model.coef_[0]),
}).sort_values("Coefficient", ascending=False)

coef

Intercept: 1.3659866464628325


,Feature,Coefficient,Odds Ratio
96,encoder__ambience_multiple,0.816052,2.261554
95,encoder__ambience_intimate,0.801391,2.228639
36,encoder__service_format_Food truck/stand,0.663859,1.942274
23,encoder__cuisine_Korean,0.604024,1.829465
94,encoder__ambience_hipster,0.593511,1.810334
...,...,...,...
16,encoder__cuisine_Chinese,-0.604629,0.546277
58,encoder__accepts_credit_cards_Yes,-0.615832,0.540191
82,encoder__noise_level_very_loud,-1.020940,0.360256
34,encoder__service_format_Buffet,-1.174309,0.309033


In [33]:
y_pred = (p_test >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred, labels=[1, 0])

cm_df = pd.DataFrame(
    cm,
    index=["Actual: High (4+)", "Actual: Not high"],
    columns=["Predicted: High (4+)", "Predicted: Not high"],
)
cm_df

,Predicted: High (4+),Predicted: Not high
Actual: High (4+),3295,1346
Actual: Not high,1333,4492


In [34]:
fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Restaurants"),
    title="Restaurant Rating Confusion Matrix (Holdout, thresholt = 0.5)"
)

fig.update_layout(
    xaxis_title="Predicted Rating Class",
    yaxis_title="Actual Rating Class"
)

fig.show()

In [35]:
results_test = pd.DataFrame({
    "Set": "Test",
    "business_id": test["business_id"],
    "stars": test["stars"],
    "Actual_high": y_test,
    "Predicted_prob_high": p_test,
    "Log_odds": log_odds_test,
    "Predicted_label": (p_test >= threshold).astype(int),
})
results_test.head().round(3)

,Set,business_id,stars,Actual_high,Predicted_prob_high,Log_odds,Predicted_label
0,Test,k0hlBqXX-Bt0vf1op7Jr1w,3.0,0,0.297,-0.861,0
1,Test,bBDDEgkFA1Otx9Lfe7BZUQ,1.5,0,0.152,-1.716,0
2,Test,eEOYSgkmpB90uNA7lDOMRA,4.0,1,0.928,2.561,1
3,Test,0bPLkL0QhhPO5kt1_EXmNQ,4.5,1,0.676,0.734,1
4,Test,kfNv-JZpuN6TVNSO6hHdkw,4.0,1,0.379,-0.494,0


In [36]:
pd.DataFrame({
    "Set": ["Train", "Test"],
    "Log loss": [log_loss(y_train, p_train), log_loss(y_test, p_test)],
    "Accuracy": [accuracy_score(y_train, p_train >= threshold),
                 accuracy_score(y_test, p_test >= threshold)]
})

,Set,Log loss,Accuracy
0,Train,0.506530,0.747524
1,Test,0.513276,0.744028


In [ ]:
thresholds = [0.10, 0.30, 0.50, 0.7, 0.90]

cost_fp = 3   # predict a high rating when the restaurant actually isn't high
cost_fn = 1    # predict not high when the restaurant actually is high

In [39]:
p_test = pipe.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, p_test)
ll = log_loss(y_test, p_test)

test_rows = []

for t in thresholds:
    y_pred = (p_test >= t).astype(int)

    tp, fn, fp, tn = confusion_matrix(
        y_test, y_pred, labels=[1, 0]
    ).ravel()

    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 0

    test_rows.append({
        "Threshold": t,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "TPR": tpr, "FPR": fpr,
        "Precision": precision, "Accuracy": accuracy, "F1": f1,
        "Total cost": cost_fp * fp + cost_fn * fn,
        "Log loss": ll, "ROC AUC": auc
    })

test_results = pd.DataFrame(test_rows)
display(test_results.round(3))

,Threshold,TP,FP,FN,TN,TPR,FPR,Precision,Accuracy,F1,Total cost,Log loss,ROC AUC
0,0.1,4572,4426,69,1399,0.985,0.760,0.508,0.571,0.670,13347,0.513,0.821
1,0.3,4126,2628,515,3197,0.889,0.451,0.611,0.700,0.724,8399,0.513,0.821
2,0.5,3295,1333,1346,4492,0.710,0.229,0.712,0.744,0.711,5345,0.513,0.821
3,0.7,2010,468,2631,5357,0.433,0.080,0.811,0.704,0.565,4035,0.513,0.821
4,0.9,406,37,4235,5788,0.087,0.006,0.916,0.592,0.160,4346,0.513,0.821


In [46]:
auc = roc_auc_score(y_test, p_test)
fpr_curve, tpr_curve, roc_thresholds = roc_curve(y_test, p_test)
roc_data = pd.DataFrame({"FPR": fpr_curve, "TPR": tpr_curve})

# FPR/TPR at each of the 5 thresholds on the test set
test_rows = []
for t in thresholds:
    pred = (p_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    test_rows.append({"threshold": t, "FPR": fp / (fp + tn), "TPR": tp / (tp + fn)})
test_results = pd.DataFrame(test_rows)

fig = px.line(roc_data, x="FPR", y="TPR",
              title=f"Restaurant High-Rating ROC Curve (Holdout) | AUC = {auc:.3f}")

fig.add_scatter(
    x=test_results["FPR"], y=test_results["TPR"],
    mode="markers+text",
    text=[f"t = {t:.1f}" for t in test_results["threshold"]],
    textposition="bottom right",
    marker=dict(size=10),
    name="Five thresholds",
)

fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines",
                line=dict(dash="dash", color="gray"), name="Random classification")

fig.update_xaxes(range=[0, 1], title="False Positive Rate (FPR)")
fig.update_yaxes(range=[0, 1], title="True Positive Rate (TPR)")
fig.show()

In [47]:
prob_true, prob_pred = calibration_curve(
    y_test, p_test, n_bins=10, strategy="uniform"
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prob_pred,
    y=prob_true,
    mode="lines+markers",
    name="Restaurant Logistic Regression"
))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    line=dict(dash="dash"),
    name="Perfect Calibration"
))

fig.update_layout(
    title="Restaurant Rating Probability Calibration",
    xaxis_title="Mean predicted probability of high rating",
    yaxis_title="Actual share of high-rated restaurants",
    width=700,
    height=500
)

fig.show()

### Regularization

In [48]:
train_ids, val_ids = train_test_split(
    train["business_id"].unique(), test_size=0.25, random_state=6021
)

tr  = train[train["business_id"].isin(train_ids)]
val = train[train["business_id"].isin(val_ids)]

X_train, y_train = tr[cats + nums],   (tr["stars"]   >= 4).astype(int)
X_val,   y_val   = val[cats + nums],  (val["stars"]  >= 4).astype(int)
X_test,  y_test  = test[cats + nums], (test["stars"] >= 4).astype(int)

display(pd.DataFrame({
    "Set": ["Training", "Validation", "Test"],
    "Restaurants": [len(X_train), len(X_val), len(X_test)],
    "High-rating proportion": [y_train.mean(), y_val.mean(), y_test.mean()],
}).round(3))

,Set,Restaurants,High-rating proportion
0,Training,31351,0.449
1,Validation,10451,0.443
2,Test,10466,0.443


In [54]:
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="infrequent_if_exist",
                              min_frequency=50), cats),
    ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                              StandardScaler()), nums),
])

In [55]:
C_values = [0.02, 0.05, 0.1, 0.3, 1, 10]
coef_rows, score_rows = [], []

for method in ["Ridge (L2)", "Lasso (L1)"]:
    for C in C_values:
        regularized = Pipeline([
            ("preprocess", clone(preprocess)),
            ("model", LogisticRegression(
                l1_ratio=0 if method == "Ridge (L2)" else 1,   # see note 2 about versions
                C=C, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123,
            )),
        ])
        regularized.fit(X_train, y_train)

        names = regularized.named_steps["preprocess"].get_feature_names_out()
        betas = regularized.named_steps["model"].coef_[0]
        for name, beta in zip(names, betas):
            coef_rows.append({"Method": method, "C": C,
                              "Feature": name.split("__")[-1], "Coefficient": beta})

        score_rows.append({"Method": method, "C": C,
                           "Val AUC": roc_auc_score(y_val, regularized.predict_proba(X_val)[:, 1]),
                           "Nonzero coefs": int((betas != 0).sum())})

coef_paths = pd.DataFrame(coef_rows)
scores = pd.DataFrame(score_rows)
display(scores.round(4))

for method in ["Ridge (L2)", "Lasso (L1)"]:
    paths = coef_paths[coef_paths["Method"] == method]
    top = paths.groupby("Feature")["Coefficient"].apply(lambda s: s.abs().max()).nlargest(15).index
    fig = px.line(
        paths[paths["Feature"].isin(top)],
        x="C", y="Coefficient", color="Feature", markers=True, log_x=True,
        title=f"Restaurants: {method} Coefficient Paths (top 15 features)",
    )
    fig.add_hline(y=0, line_dash="dash")
    fig.show()

,Method,C,Val AUC,Nonzero coefs
0,Ridge (L2),0.02,0.8276,130
1,Ridge (L2),0.05,0.8283,130
2,Ridge (L2),0.10,0.8285,130
3,Ridge (L2),0.30,0.8286,130
4,Ridge (L2),1.00,0.8286,130
5,Ridge (L2),10.00,0.8286,130
6,Lasso (L1),0.02,0.8235,61
7,Lasso (L1),0.05,0.8269,82
8,Lasso (L1),0.10,0.8282,100
9,Lasso (L1),0.30,0.8286,113


In [56]:
coef_paths["Nonzero"] = coef_paths["Coefficient"].abs() > 1e-6
demo_counts = coef_paths.groupby(["C", "Method"])["Nonzero"].sum().unstack()
display(demo_counts.rename_axis(columns=None))

,Lasso (L1),Ridge (L2)
C,,
0.02,61,130
0.05,82,130
0.10,100,130
0.30,113,130
1.00,122,130
10.00,125,130


In [57]:
full_model = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        C=np.inf, solver="lbfgs", max_iter=5000, tol=1e-8, random_state=123,
    )),
])
full_model.set_params(preprocess__numeric__simpleimputer__add_indicator=False)

full_model.fit(X_train, y_train)
print("Validation AUC:", roc_auc_score(y_val, full_model.predict_proba(X_val)[:, 1]))

Validation AUC: 0.8277468456171062


In [58]:
ridge = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        l1_ratio=0, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123
    ))
])

lasso = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        l1_ratio=1, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123
    ))
])

In [63]:
# faster solvers, same models
ridge.set_params(model__solver="lbfgs")
lasso.set_params(model__solver="saga", model__tol=1e-3, model__max_iter=2000)

C_grid = np.unique(np.r_[np.logspace(-3, 1, 12), C_values])   # about 15 values instead of 41
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=6021)

ridge_cv = GridSearchCV(ridge, {"model__C": C_grid}, cv=cv,
                        scoring="neg_log_loss", error_score="raise", n_jobs=-1)
ridge_cv.fit(X_train, y_train)

lasso_cv = GridSearchCV(lasso, {"model__C": C_grid}, cv=cv,
                        scoring="neg_log_loss", error_score="raise", n_jobs=-1)
lasso_cv.fit(X_train, y_train)

for name, search in [("Ridge", ridge_cv), ("Lasso", lasso_cv)]:
    print(f"{name}: best C = {search.best_params_['model__C']:.4g}, "
          f"CV log loss = {-search.best_score_:.4f}")

/Users/anhly/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/anhly/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/anhly/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/Users/anhly/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] durin

Ridge: best C = 0.152, CV log loss = 0.5119
Lasso: best C = 1.874, CV log loss = 0.5120


In [64]:
ridge_model = ridge_cv.best_estimator_
lasso_model = lasso_cv.best_estimator_

best_C_ridge = ridge_cv.best_params_["model__C"]
best_C_lasso = lasso_cv.best_params_["model__C"]

models = {"All 12 (unpenalized)": full_model,
          "Ridge": ridge_model, "Lasso": lasso_model}
searches = {"Ridge": ridge_cv, "Lasso": lasso_cv}

display(pd.DataFrame({
    "Model": ["Ridge", "Lasso"],
    "Selected C": [best_C_ridge, best_C_lasso],
    "Mean CV log loss": [-ridge_cv.best_score_, -lasso_cv.best_score_]
}).round(6))

,Model,Selected C,Mean CV log loss
0,Ridge,0.151991,0.511862
1,Lasso,1.873817,0.512045


In [65]:
cv_tables = []
for name, search in searches.items():
    results = pd.DataFrame(search.cv_results_)
    cv_tables.append(pd.DataFrame({
        "Model": name,
        "C": results["param_model__C"].astype(float),
        "CV log loss": -results["mean_test_score"],
        "Fold SD": results["std_test_score"]
    }))

cv_results = pd.concat(cv_tables, ignore_index=True)

In [66]:
for name, search in searches.items():
    chosen_C = search.best_params_["model__C"]
    fig = px.line(
        cv_results[cv_results["Model"] == name],
        x="C", y="CV log loss", log_x=True, markers=True,
        title=f"{name}: five-fold CV log loss | selected C = {chosen_C:g}"
    )
    fig.add_vline(x=chosen_C, line_dash="dash", line_color="black")
    fig.update_xaxes(tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                     ticktext=["0.0001", "0.001", "0.01", "0.1", "1", "10", "100", "1000"])
    fig.show()

In [112]:
C_lasso_sparse = 0.01

lasso_sparse = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        C=C_lasso_sparse, l1_ratio=1, solver="liblinear",
        max_iter=5000, random_state=123,
    )),
])
lasso_sparse.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](45,)","['state','cuisine','service_format',...,'n_cuisine_groups', 'category_count','chain_size']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,45
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remain

In [113]:
models = {
    "Full Model": full_model,
    "Ridge": ridge_cv.best_estimator_,       # C = 0.152
    "Lasso": lasso_cv.best_estimator_,       # C = 1.874
    "Lasso (C = 0.01)": lasso_sparse,        # remove this line if you deleted the cell
}

In [114]:
coef_tables, model_rows = [], []

for name, model in models.items():
    names = [f.split("__")[-1] for f in model.named_steps["preprocess"].get_feature_names_out()]
    beta = model.named_steps["model"].coef_.ravel()
    coef_tables.append(pd.Series(beta, index=names, name=name))
    model_rows.append({"Model": name, "Nonzero slopes": int((abs(beta) > 1e-6).sum()),
                       "Total features": len(beta)})

coefficients = pd.concat(coef_tables, axis=1).rename_axis("Feature").reset_index()

display(coefficients.round(6))
display(pd.DataFrame(model_rows))

print("Lasso drops:",
      coefficients.loc[coefficients["Lasso"].abs() <= 1e-6, "Feature"].tolist())

,Feature,Full Model,Ridge,Lasso,Lasso (C = 0.01)
0,state_AZ,-1.482307,-0.047550,0.230950,0.000000
1,state_CA,-1.090850,0.008375,0.147460,0.000000
2,state_DE,-1.967810,-0.169296,0.437844,0.000000
3,state_FL,-2.339305,0.031963,0.717767,0.000000
4,state_ID,-0.808728,-0.205986,-0.146658,0.000000
...,...,...,...,...,...
125,category_count,0.121368,0.113947,0.116585,0.099362
126,chain_size,-0.434909,-0.444027,-0.441753,-0.510867
127,missingindicator_listed_days,NaN,0.120006,-0.234650,0.000000
128,missingindicator_weekly_hours,NaN,0.151334,0.152918,0.118346


,Model,Nonzero slopes,Total features
0,Full Model,127,127
1,Ridge,130,130
2,Lasso,126,130
3,Lasso (C = 0.01),46,130


Lasso drops: ['state_NV', 'state_infrequent_sklearn', 'wifi_paid', 'attire_dressy_formal']


In [115]:
C_path = np.unique(np.r_[C_grid, C_values])
path_rows = []

for name, search in searches.items():
    for C in C_path:
        path_model = clone(search.best_estimator_).set_params(model__C=C)
        path_model.fit(X_train, y_train)
        beta = path_model.named_steps["model"].coef_[0]
        for feature, value in zip(feature_names, beta):
            path_rows.append({"Model": name, "C": C,
                              "Feature": feature, "Coefficient": value})

paths = pd.DataFrame(path_rows)

In [116]:
for name, search in searches.items():
    chosen_C = search.best_params_["model__C"]
    fig = px.line(
        paths[paths["Model"] == name], x="C", y="Coefficient",
        color="Feature", log_x=True,
        category_orders={"Feature": feature_names},
        title=f"{name}: coefficient paths | CV-selected C = {chosen_C:g}"
    )
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_vline(x=chosen_C, line_dash="dash", line_color="black")
    fig.update_layout(height=500)
    fig.update_xaxes(tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                     ticktext=["0.0001", "0.001", "0.01", "0.1", "1", "10", "100", "1000"])
    fig.show()

In [117]:
sparsity_rows = []
for C in [0.01, 0.02, best_C_lasso]:
    demo_lasso = clone(lasso_model).set_params(model__C=C).fit(X_train, y_train)
    beta = demo_lasso.named_steps["model"].coef_[0]
    grid_match = cv_results[(cv_results["Model"] == "Lasso") & np.isclose(cv_results["C"], C)]
    cv_loss = grid_match["CV log loss"].iloc[0] if len(grid_match) else np.nan
    sparsity_rows.append({"C": C, "Nonzero slopes": int((abs(beta) > 1e-6).sum()),
                          "Mean CV log loss": cv_loss,
                          "Role": "CV optimum" if C == best_C_lasso else "Stronger-penalty example"})

display(pd.DataFrame(sparsity_rows).round(6))

,C,Nonzero slopes,Mean CV log loss,Role
0,0.010000,46,NaN,Stronger-penalty example
1,0.020000,61,0.521269,Stronger-penalty example
2,1.873817,122,0.512045,CV optimum


In [118]:
cost_fp, cost_fn = 3, 1
thresholds = np.round(np.arange(0.00, 1.01, 0.01), 2)
p_val = {name: model.predict_proba(X_val)[:, 1] for name, model in models.items()}

threshold_rows = []
for name, probabilities in p_val.items():
    for t in thresholds:
        predicted = (probabilities >= t).astype(int)
        tp, fn, fp, tn = confusion_matrix(y_val, predicted, labels=[1, 0]).ravel()
        threshold_rows.append({"Model": name, "Threshold": t,
                               "Validation cost": cost_fp * fp + cost_fn * fn,
                               "Validation accuracy": (tp + tn) / len(y_val)})

validation_results = pd.DataFrame(threshold_rows)

In [119]:
best_thresholds = {}
selected_rows = []

for name in models:
    results = validation_results[validation_results["Model"] == name].copy()
    results["Distance from 0.5"] = abs(results["Threshold"] - 0.5)
    best = results.sort_values(["Validation cost", "Distance from 0.5", "Threshold"]).iloc[0]
    best_thresholds[name] = float(best["Threshold"])
    selected_rows.append(best)

threshold_summary = pd.DataFrame(selected_rows).drop(columns="Distance from 0.5")
display(threshold_summary.round(4))

,Model,Threshold,Validation cost,Validation accuracy
74,Full Model,0.74,3909,0.6882
173,Ridge,0.72,3892,0.6980
275,Lasso,0.73,3893,0.6937
376,Lasso (C = 0.01),0.73,4011,0.6621


In [120]:
fig = px.line(
    validation_results, x="Threshold", y="Validation cost", color="Model",
    title="Choose thresholds on validation data: cost = 3 × FP + FN"
)
for name, t in best_thresholds.items():
    point = threshold_summary[threshold_summary["Model"] == name]
    fig.add_scatter(x=[t], y=point["Validation cost"], mode="markers",
                    marker=dict(size=11, symbol="diamond"),
                    name=f"{name}: selected t = {t:.2f}")
fig.show()

In [121]:
p_test = {name: model.predict_proba(X_test)[:, 1] for name, model in models.items()}
feature_counts = pd.DataFrame(model_rows)
test_comparison = pd.DataFrame({
    "Model": list(models),
    "Test log loss": [log_loss(y_test, p_test[name]) for name in models],
    "Nonzero slopes": feature_counts["Nonzero slopes"],
    "Selected threshold": [best_thresholds[name] for name in models]
})
display(test_comparison.round(6))

,Model,Test log loss,Nonzero slopes,Selected threshold
0,Full Model,0.514138,127,0.74
1,Ridge,0.514053,130,0.72
2,Lasso,0.513947,126,0.73
3,Lasso (C = 0.01),0.530139,46,0.73


In [122]:
logloss_figure = px.bar(
    test_comparison, x="Model", y="Test log loss", text_auto=".4f",
    title="Test log loss — lower is better"
)
logloss_figure.update_traces(marker_color="#234C6A", width=0.55, textposition="outside")
logloss_figure.update_layout(showlegend=False, xaxis_title=None, height=420)
logloss_figure.update_xaxes(showgrid=False)
logloss_figure.update_yaxes(showgrid=False, range=[0, test_comparison["Test log loss"].max() * 1.16])
logloss_figure.show()

In [124]:
model_names = list(models)
labels = ["Win", "Loss"]
colors = [[0, "#F6F9FC"], [1, "#9EC1DF"]]
titles = ([f"{name}<br>t = 0.50" for name in model_names] +
          [f"{name}<br>selected t = {best_thresholds[name]:.2f}" for name in model_names])
selected_cm_figure = make_subplots(rows=2, cols=4, subplot_titles=titles,
                                  vertical_spacing=0.25, horizontal_spacing=0.10)

for row in [1, 2]:
    for col, name in enumerate(model_names, start=1):
        t = 0.50 if row == 1 else best_thresholds[name]
        cm = confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0])
        selected_cm_figure.add_trace(go.Heatmap(
            z=cm, x=labels, y=labels, zmin=0, zmax=int(y_test.value_counts().max()),
            colorscale=colors, showscale=False, texttemplate="%{z}",
            textfont=dict(color="#132F4C", size=18),
            hovertemplate="Actual %{y}<br>Predicted %{x}<br>Count %{z}<extra></extra>"
        ), row=row, col=col)

selected_cm_figure.update_xaxes(title_text="Predicted")
selected_cm_figure.update_yaxes(title_text="Actual", autorange="reversed")
selected_cm_figure.update_layout(template="plotly_white", height=690,
                                title="Test confusion matrices: default and validation-selected thresholds")
selected_cm_figure.show()

In [125]:
test_cms = {}
for name in model_names:
    test_cms[name] = [confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0])
                      for t in thresholds]

threshold_cm_figure = make_subplots(rows=1, cols=4, subplot_titles=model_names,
                                   horizontal_spacing=0.10)
for col, name in enumerate(model_names, start=1):
    threshold_cm_figure.add_trace(go.Heatmap(
        z=test_cms[name][50], x=labels, y=labels,
        zmin=0, zmax=int(y_test.value_counts().max()), colorscale=colors,
        showscale=False, texttemplate="%{z}", textfont=dict(color="#132F4C", size=18),
        hovertemplate="Actual %{y}<br>Predicted %{x}<br>Count %{z}<extra></extra>"
    ), row=1, col=col)

steps = []
for i, t in enumerate(thresholds):
    steps.append(dict(method="update", label=f"{t:.2f}",
                      args=[{"z": [test_cms[name][i].tolist() for name in model_names]}]))

threshold_cm_figure.update_xaxes(title_text="Predicted")
threshold_cm_figure.update_yaxes(title_text="Actual", autorange="reversed")
threshold_cm_figure.update_layout(
    template="plotly_white", height=470,
    title="Test confusion matrices: explore every searched threshold",
    sliders=[dict(active=50, steps=steps, pad=dict(t=50),
                  currentvalue=dict(prefix="Common threshold = "))]
)
threshold_cm_figure.show()

In [126]:
metric_rows = []
for name in model_names:
    show_thresholds = sorted(set([0.10, 0.30, 0.50, 0.70, 0.90, best_thresholds[name]]))
    for t in show_thresholds:
        tp, fn, fp, tn = confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0]).ravel()
        metric_rows.append({
            "Model": name, "Threshold": t, "Selected on validation": t == best_thresholds[name],
            "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "Accuracy": (tp + tn) / len(y_test), "Recall": tp / (tp + fn),
            "Precision": tp / (tp + fp) if tp + fp else 0,
            "Cost": cost_fp * fp + cost_fn * fn,
            "Log loss": log_loss(y_test, p_test[name])
        })

test_threshold_results = pd.DataFrame(metric_rows)
display(test_threshold_results.round(4)) 

,Model,Threshold,Selected on validation,TP,FN,FP,TN,Accuracy,Recall,Precision,Cost,Log loss
0,Full Model,0.10,False,4572,69,4452,1373,0.5680,0.9851,0.5066,13425,0.5141
1,Full Model,0.30,False,4145,496,2641,3184,0.7003,0.8931,0.6108,8419,0.5141
2,Full Model,0.50,False,3309,1332,1349,4476,0.7438,0.7130,0.7104,5379,0.5141
3,Full Model,0.70,False,2031,2610,465,5360,0.7062,0.4376,0.8137,4005,0.5141
4,Full Model,0.74,True,1691,2950,347,5478,0.6850,0.3644,0.8297,3991,0.5141
5,Full Model,0.90,False,401,4240,37,5788,0.5913,0.0864,0.9155,4351,0.5141
6,Ridge,0.10,False,4576,65,4486,1339,0.5652,0.9860,0.5050,13523,0.5141
7,Ridge,0.30,False,4140,501,2648,3177,0.6991,0.8920,0.6099,8445,0.5141
8,Ridge,0.50,False,3301,1340,1347,4478,0.7433,0.7113,0.7102,5381,0.5141
9,Ridge,0.70,False,1995,2646,472,5353,0.7021,0.4299,0.8087,4062,0.5141


In [127]:
example = test.sample(1, random_state=6021)          # or test[test["name"] == "Some Name"].head(1)
recent_restaurant = example[cats + nums]
actual_high = int(example["stars"].iloc[0] >= 4)

display(example[["name", "city", "state", "cuisine", "service_format", "price_range", "stars"]])


,name,city,state,cuisine,service_format,price_range,stars
4570,Mediterranean Cafe & Ice Cream,Philadelphia,PA,Breakfast & Cafe,Standard,2,2.0


In [128]:
prediction_rows = []
for name, model in models.items():
    probability = model.predict_proba(recent_restaurant)[0, 1]
    t = best_thresholds[name]
    prediction_rows.append({
        "Model": name,
        "P(high rating)": probability,
        "High-rating probability (%)": 100 * probability,
        "Validation-selected threshold": t,
        "Prediction": "High (4+)" if probability >= t else "Not high",
        "Actual": "High (4+)" if actual_high else "Not high",
    })

recent_predictions = pd.DataFrame(prediction_rows)
display(recent_predictions.round(4))
print(f"Actual: {example['name'].iloc[0]} has {example['stars'].iloc[0]} stars.")

,Model,P(high rating),High-rating probability (%),Validation-selected threshold,Prediction,Actual
0,Full Model,0.2622,26.2218,0.74,Not high,Not high
1,Ridge,0.2707,27.0675,0.72,Not high,Not high
2,Lasso,0.2706,27.0598,0.73,Not high,Not high
3,Lasso (C = 0.01),0.3868,38.6762,0.73,Not high,Not high


Actual: Mediterranean Cafe & Ice Cream has 2.0 stars.


In [129]:
new_restaurant = pd.DataFrame({c: ["Unknown"] for c in cats} | {c: [np.nan] for c in nums})
new_restaurant.update(pd.DataFrame({
    "state": ["PA"], "latitude": [39.95], "longitude": [-75.16],      # Philadelphia
    "cuisine": ["Thai"], "service_format": ["Standard"], "is_chain": ["No"],
    "chain_size": [1], "n_cuisine_groups": [1], "category_count": [3],
    "price_range": ["2"], "takeout": ["Yes"], "delivery": ["Yes"],
    "reservations": ["Yes"], "alcohol": ["beer_and_wine"], "noise_level": ["average"],
    "listed_days": [6], "weekly_hours": [60], "avg_daily_hours": [10],
    "open_weekend": ["Yes"], "late_night": ["No"], "early_open": ["No"],
}))
new_restaurant = new_restaurant[cats + nums]
new_restaurant.head(5)

,state,cuisine,service_format,is_chain,price_range,takeout,delivery,outdoor_seating,reservations,has_tv,...,late_night,early_open,latitude,longitude,listed_days,weekly_hours,avg_daily_hours,n_cuisine_groups,category_count,chain_size
0,PA,Thai,Standard,No,2,Yes,Yes,Unknown,Yes,Unknown,...,No,No,39.95,-75.16,6.0,60.0,10.0,1.0,3.0,1.0


### Summary